# Entrenamiento en Colab con GPU (basado en `comandos_reducidos_actualizado.txt`)

Este notebook ejecuta solo los comandos **activos** (fuera de bloques entre `"""`).

- `01_matrix_factorization.py`: sin comandos activos (todo marcado como ya ejecutado)
- `02_two_tower.py`: comandos activos incluidos
- `03_two_tower_wide_deep.py`: comandos activos incluidos
- `04_sasrec.py`: comandos activos incluidos


## 1) Configura runtime GPU en Colab
En Colab: `Runtime` -> `Change runtime type` -> `T4/A100/L4 (GPU)`.


In [ ]:
import subprocess
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

nvidia_smi = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True)
if nvidia_smi.returncode == 0:
    print(nvidia_smi.stdout)
else:
    print('nvidia-smi no disponible en este runtime.')

if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        'No hay GPU activa. En Colab ve a Runtime > Change runtime type > Hardware accelerator = GPU, guarda y ejecuta de nuevo.'
    )


## 2) Repo y dependencias
Ajusta `PROJECT_DIR` a la ruta real en tu Colab. Si aún no has clonado el repo, descomenta la línea de `git clone`.


In [ ]:
from pathlib import Path

# Descomenta y rellena si necesitas clonar en Colab:
# !git clone https://github.com/<TU_USUARIO>/<TU_REPO>.git /content/pfg

PROJECT_DIR = Path('/content/pfg/pfg-models')

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'No existe {PROJECT_DIR}. Clona/monta el repo y vuelve a ejecutar esta celda.')

%cd {PROJECT_DIR}
!pip -q install -r requirements.txt


## 3) (Opcional) Login de W&B


In [ ]:
import os
from getpass import getpass

if not os.environ.get('WANDB_API_KEY'):
    os.environ['WANDB_API_KEY'] = getpass('WANDB_API_KEY (Enter para saltar): ')

if os.environ.get('WANDB_API_KEY'):
    !wandb login --relogin $WANDB_API_KEY
else:
    print('Sin login de W&B (se ejecutará sin tracking remoto si tus scripts lo permiten).')


## 4) Helpers y comandos activos
Se añade `--device cuda` automáticamente a modelos `02`, `03` y `04` (si no estaba ya en el comando).


In [ ]:
%cd /content/pfg/pfg-models/src/models

import subprocess
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device objetivo:', DEVICE)
if DEVICE != 'cuda':
    raise RuntimeError('Este notebook esta preparado para GPU. Activa GPU en Colab antes de entrenar.')

def adapt_command(cmd: str) -> str:
    if cmd.startswith(('python 02_', 'python 03_', 'python 04_')) and '--device' not in cmd:
        return f"{cmd} --device {DEVICE}"
    return cmd

def run_many(commands):
    for i, cmd in enumerate(commands, 1):
        final_cmd = adapt_command(cmd.strip())
        print(f"\n[{i}/{len(commands)}] {final_cmd}")
        subprocess.run(final_cmd, shell=True, check=True)

CMDS_02 = [
    "python 02_two_tower.py --dataset movielens --dim 128 --lr 0.001 --dropout 0.1 --n-neg 5 --batch 1024 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group movielens-tt-final",
    "python 02_two_tower.py --dataset amazon_electronics --dim 64 --lr 0.001 --dropout 0.1 --n-neg 5 --batch 1024 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group amazon_electronics-tt-final",
    "python 02_two_tower.py --dataset yelp --sample 3000000 --dim 64 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group yelp-tt-validate",
    "python 02_two_tower.py --dataset yelp --sample 3000000 --dim 128 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group yelp-tt-validate",
    "python 02_two_tower.py --dataset yelp --dim 128 --lr 0.001 --dropout 0.1 --n-neg 5 --batch 1024 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group yelp-tt-final",
    "python 02_two_tower.py --dataset lastfm --sample 3000000 --dim 64 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group lastfm-tt-validate",
    "python 02_two_tower.py --dataset lastfm --sample 3000000 --dim 128 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group lastfm-tt-validate",
    "python 02_two_tower.py --dataset lastfm --dim 128 --lr 0.001 --dropout 0.1 --n-neg 5 --batch 1024 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group lastfm-tt-final",
    "python 02_two_tower.py --dataset foursquare --sample 3000000 --dim 64 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group foursquare-tt-validate",
    "python 02_two_tower.py --dataset foursquare --sample 3000000 --dim 128 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group foursquare-tt-validate",
    "python 02_two_tower.py --dataset foursquare --dim 128 --lr 0.001 --dropout 0.1 --n-neg 5 --batch 1024 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group foursquare-tt-final",
]

CMDS_03 = [
    "python 03_two_tower_wide_deep.py --dataset movielens --sample 1000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group movielens-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset movielens --sample 3000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 5 --wandb --wandb-project pfg-recs --wandb-group movielens-ttwd-final",
    "python 03_two_tower_wide_deep.py --dataset amazon_electronics --sample 1000000 --candidates 500 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group amazon_electronics-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset amazon_electronics --sample 1000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group amazon_electronics-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset amazon_electronics --sample 3000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 5 --wandb --wandb-project pfg-recs --wandb-group amazon_electronics-ttwd-final",
    "python 03_two_tower_wide_deep.py --dataset yelp --sample 1000000 --candidates 500 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group yelp-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset yelp --sample 1000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group yelp-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset yelp --sample 3000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 5 --wandb --wandb-project pfg-recs --wandb-group yelp-ttwd-final",
    "python 03_two_tower_wide_deep.py --dataset foursquare --sample 1000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group foursquare-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset foursquare --sample 3000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 5 --wandb --wandb-project pfg-recs --wandb-group foursquare-ttwd-final",
]

CMDS_04 = [
    "python 04_sasrec.py --dataset movielens --max-len 100 --d-model 128 --n-heads 2 --n-layers 2 --dropout 0.1 --neg-samples 5 --lr 0.001 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group movielens-sasrec-final",
    "python 04_sasrec.py --dataset amazon_electronics --max-len 100 --d-model 128 --n-heads 2 --n-layers 2 --dropout 0.1 --neg-samples 5 --lr 0.001 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group amazon_electronics-sasrec-final",
    "python 04_sasrec.py --dataset yelp --max-len 100 --d-model 128 --n-heads 2 --n-layers 2 --dropout 0.1 --neg-samples 5 --lr 0.001 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group yelp-sasrec-final",
    "python 04_sasrec.py --dataset foursquare --max-len 100 --d-model 128 --n-heads 2 --n-layers 2 --dropout 0.1 --neg-samples 5 --lr 0.001 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group foursquare-sasrec-final",
]

print('Comandos cargados:', len(CMDS_02) + len(CMDS_03) + len(CMDS_04))


## 5) Ejecutar modelo 02 (Two Tower)


In [ ]:
run_many(CMDS_02)


## 6) Ejecutar modelo 03 (Two Tower Wide & Deep)


In [ ]:
run_many(CMDS_03)


## 7) Ejecutar modelo 04 (SASRec)


In [ ]:
run_many(CMDS_04)
